# Live-pool validation — does the re-anchored cite-budget model earn its keep?

The verdict notebook for the two Backlog tickets under *Citations & graph data*
(the live-path re-anchoring ticket and the corpus-models ticket). The collector
(`src/ml_pipelines/live_pool_validation/collect.py`, run on the corpus machine)
simulated, for every `cite_budget` seed, **the exact pool the live S2 fallback
would hold** — the newest ~9k citers — and recorded the exact rules and both
model anchorings side by side. Three questions, one column pair each:

1. **Null hypothesis (re-anchoring):** does `model_pool_anchor` (age measured
   from the oldest reachable citer) just track `n_star_live` (the label we can
   compute exactly at serve time)? If yes, the model is redundant on the live
   path — see `docs/predict-vs-compute.md`.
2. **The broken baseline:** `model_seed_anchor` vs `n_star_live` — the
   pre-v5.5.0 behavior, expected to overshoot on truncated old seeds.
3. **Corpus ticket:** does `model_seed_anchor` track `n_star_corpus` (the label
   over the corpus's full ranked pool)? The model's premise *should* hold there
   — this measures it.

Plus an eyeball table of the anchors' `band_start` (the latest-gap tau rule on
truncated-pool landmarks — fit on whole-history distributions, so the transfer
is checked by eye, not assumed).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CORPUS = Path("../../src/ml_pipelines/live_pool_validation/corpus.csv")
if not CORPUS.exists():
    raise FileNotFoundError(
        f"{CORPUS} not collected yet — run "
        "`uv run python -m ml_pipelines.live_pool_validation.collect` "
        "on the machine that holds the ingested corpus, commit corpus.csv, "
        "then re-run this notebook."
    )
corpus = pd.read_csv(CORPUS)
print(f"{len(corpus)} seeds ({int(corpus['truncated'].sum())} truncated, "
      f"{int(corpus['is_anchor'].sum())} anchors)")
corpus.head()

## The anchors, side by side

The four working examples first — if a number here looks absurd, no aggregate
metric below can redeem it (the OpenAlex-misdate lesson).

In [ ]:
ANCHOR_COLUMNS = [
    "label", "s2_year", "s2_citation_count", "citer_count", "truncated",
    "oldest_pool_year", "n_star_live", "banded_live",
    "model_pool_anchor", "model_seed_anchor", "n_star_corpus",
    "band_start", "landmark_max_year",
]
corpus[corpus["is_anchor"] == 1][ANCHOR_COLUMNS]

## Q1 + Q2 — both anchorings against the computable live label

Each point is a seed; the diagonal is a perfect prediction of the label the
serve path can compute exactly. The re-anchored model has to hug the diagonal
*better than the seed-anchored one* to be an improvement — and if it hugs it
closely, it's redundant (the null hypothesis): the exact rule needs no
predictor.

In [ ]:
def diagonal_fit(observed: pd.Series, predicted: pd.Series) -> float:
    """R² of `predicted` against the y=x diagonal (not a refit line)."""
    mask = observed.notna() & predicted.notna()
    residual = ((predicted[mask] - observed[mask]) ** 2).sum()
    total = ((observed[mask] - observed[mask].mean()) ** 2).sum()
    return 1 - residual / total if total else float("nan")


figure, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for axis, column, title in [
    (axes[0], "model_pool_anchor", "re-anchored (oldest reachable citer)"),
    (axes[1], "model_seed_anchor", "seed-anchored (pre-v5.5.0 baseline)"),
]:
    truncated = corpus["truncated"] == 1
    axis.scatter(corpus.loc[~truncated, "n_star_live"], corpus.loc[~truncated, column],
                 label="full history reachable", alpha=0.7)
    axis.scatter(corpus.loc[truncated, "n_star_live"], corpus.loc[truncated, column],
                 label="truncated pool", alpha=0.9, marker="s")
    limit = max(corpus["n_star_live"].max(), corpus[column].max()) * 1.05
    axis.plot([0, limit], [0, limit], linestyle=":", linewidth=1)
    axis.set_title(f"{title}\nR² vs diagonal: "
                   f"{diagonal_fit(corpus['n_star_live'], corpus[column]):.2f}")
    axis.set_xlabel("n*_live (exact label on the truncated pool)")
    axis.legend()
axes[0].set_ylabel("model prediction")
figure.tight_layout()

## Q3 — the corpus ticket: the model on full-history corpus pools

The model's premise (labels collected over whole-history rankings) *should*
transfer from OpenAlex pools to corpus pools. This is the measurement the
corpus ticket asked for: seed-anchored predictions against `n_star_corpus`.

In [ ]:
figure, axis = plt.subplots(figsize=(6, 5))
axis.scatter(corpus["n_star_corpus"], corpus["model_seed_anchor"], alpha=0.7)
for _, anchor in corpus[corpus["is_anchor"] == 1].iterrows():
    axis.annotate(anchor["label"], (anchor["n_star_corpus"], anchor["model_seed_anchor"]),
                  fontsize=8, xytext=(4, 4), textcoords="offset points")
limit = max(corpus["n_star_corpus"].max(), corpus["model_seed_anchor"].max()) * 1.05
axis.plot([0, limit], [0, limit], linestyle=":", linewidth=1)
axis.set_xlabel("n*_corpus (exact label on the full corpus pool)")
axis.set_ylabel("model_seed_anchor")
axis.set_title(f"R² vs diagonal: "
               f"{diagonal_fit(corpus['n_star_corpus'], corpus['model_seed_anchor']):.2f}")
figure.tight_layout()

## Verdict

*(Filled in after the collection runs — the framing to answer against:)*

- **Q1/Q2:** if `model_pool_anchor` tracks `n_star_live` closely, the null
  hypothesis stands — the live path keeps the exact banded selection and the
  model stays off it (`docs/predict-vs-compute.md`); the re-anchoring is then
  recorded as validated-but-unneeded. If it *diverges* while the seed-anchored
  baseline diverges worse, re-anchoring is a real repair wherever a pre-fetch
  estimate is needed.
- **Q3:** if the diagonal R² is healthy and the anchors sane, the corpus path
  can keep `model_budget` as its landmark total; if not, the corpus ticket's
  fallback applies (run `density_selection` over a corpus pool — it's local
  too).
- **Band starts:** eyeball the anchors' `band_start` against where their
  landmark clusters actually thin out; a boundary pinned to the `max_span`
  floor on every truncated seed would mean the tau rule doesn't transfer.